In [ ]:
import at3d
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import mayavi.mlab as mlab
import scipy.io as sio
import os
import logging
from collections import OrderedDict
import xarray as xr
import random
from scipy.ndimage import affine_transform
import copy
from scipy.ndimage import binary_erosion

import sys
sys.path.append('../CloudCT_utils/')

from CloudCTUtils import *

# %matplotlib qt
# commenc all mlab.show() when using qt

# Prepare CloudCT simple string of pearls setup:10 satellites with 100km distance (on orbit arc).

In [3]:
# orbit altitude:
Rsat = 500 # km
GSD = 0.5*0.02 # in km, it is the ground spatial resolution.
wavelengths_micron = 0.672  #0.672 , 1.6
sun_azimuth = 30
sun_zenith = 155
maxiter = 150
n_jobs = 60
SATS_NUMBER_SETUP = 10 # satellites number to build the setup, for the inverse, we can use less satellites.
solarmu = np.cos(np.deg2rad(sun_zenith)) # Note that `solarmu`=0.0 (purely horizontal solar beam) is not permitted.


IFVISUALIZE = True
scale = 50
axisWidth = 2.0
axisLenght=50

# How to rotate the only satellites: (not sun and cloud)


In [4]:
theta = 45 # random.randint(0, 180)
theta_rad = np.deg2rad(theta)

# Prepare sources:

In [5]:
# sun:
source = at3d.source.solar(wavelength=wavelengths_micron,\
                           solarmu=solarmu,\
                           solar_azimuth=sun_azimuth,\
                           solarflux=1.0, skyrad=0.0)

#### Compute sun_direction ####
SUN_THETA = np.deg2rad(sun_zenith)
SUN_PHI = np.deg2rad(sun_azimuth)
# calc sun direction from lookat to sun:
sun_x = np.sin(SUN_THETA) * np.cos(SUN_PHI)
sun_y = np.sin(SUN_THETA) * np.sin(SUN_PHI)
sun_z = np.cos(SUN_THETA)
sun_direction = np.array([sun_x, sun_y, sun_z])

sun_virtual_position = -sun_direction*2


# Prepare mediumes:

In [6]:
#---------------------------------------------
#---------------------------------------------
# -----------------------------------------------
# --------- Define geometry----------------------
# -----------------------------------------------
dx = 0.02
dy = 0.02
dz = 0.02

nz = 32
nx = 50
ny = 50

# grid:
xgrid = np.linspace(0, float_round(dx*nx)-dx, nx)
ygrid = np.linspace(0, float_round(dy*ny)-dy, ny)
zgrid = np.linspace(0, float_round(dz*nz)-dz, nz)

X, Y, Z = np.meshgrid(xgrid, ygrid, zgrid, indexing='ij')

B = ((X - (0.5*nx*dx))**2)/0.8 + ((Z - (0.5*nz*dz))**2)*5 + ((Y - (0.5*ny*dy))**2)*5
A = (B<(0.3*nx*dx)**2)

#USED FOV, RESOLUTION and SAT_LOOKATS:
PIXEL_FOOTPRINT = GSD # km
L = max(xgrid.max() + dx, ygrid.max() + dy)

fov = 2*np.rad2deg(np.arctan(0.5*L/(Rsat)))
cny = int(np.floor(L/PIXEL_FOOTPRINT))
cnx = int(0.7*np.floor(L/PIXEL_FOOTPRINT))

CENTER_OF_MEDIUM_BOTTOM = [0.5*nx*dx , 0.5*ny*dy , 0]
# Somtimes it is more convinent to use wide fov to see the whole cloud
# from all the view points. so the FOV is aslo tuned:
IFTUNE_CAM = True
# --- TUNE FOV, CNY,CNX:
if(IFTUNE_CAM):
    L = 1.2*L
    fov = 2*np.rad2deg(np.arctan(0.5*L/(Rsat)))
    cny = int(np.floor(L/PIXEL_FOOTPRINT))
    cnx = int(0.7*np.floor(L/PIXEL_FOOTPRINT))    

# not for all the mediums the CENTER_OF_MEDIUM_BOTTOM is a good place to lookat.
# tuning is applied by the variavle LOOKAT.
LOOKAT = CENTER_OF_MEDIUM_BOTTOM
if(IFTUNE_CAM):
    LOOKAT[2] = 0.68*nz*dz # tuning. if IFTUNE_CAM = False, just lookat the bottom

SAT_LOOKATS = np.array(SATS_NUMBER_SETUP*LOOKAT).reshape(-1,3)# currently, all satellites lookat the same point.

print(20*"-")
print(20*"-")
print(20*"-")

print("CAMERA intrinsics summary")
print("fov = {}[deg], cnx = {}[pixels],cny ={}[pixels]".format(fov,cnx,cny))

print(20*"-")
print(20*"-")
print(20*"-")


#A[:,:,0:2] = 0
alfa = 0.6
lwc = alfa*A
veff = 0.1*A
reff = 10*A

--------------------
--------------------
--------------------
CAMERA intrinsics summary
fov = 0.13750980482671657[deg], cnx = 84[pixels],cny =120[pixels]
--------------------
--------------------
--------------------


Rotate & interpulate:

In [7]:
cos_theta = np.cos(theta_rad)
sin_theta = np.sin(theta_rad)
ROT_Z =  np.array([
    [cos_theta, -sin_theta, 0,0],
    [sin_theta, cos_theta, 0,0],
    [0, 0, 1,0],
    [0, 0, 0,1]
])

T = np.array([
      [1, 0, 0, LOOKAT[0]],
      [0, 1, 0, LOOKAT[1]],
      [0, 0, 1, LOOKAT[2]],
      [0, 0, 0, 1]
  ])

T_inv = np.array([
    [1, 0, 0, -LOOKAT[0]],
    [0, 1, 0, -LOOKAT[1]],
    [0, 0, 1, -LOOKAT[2]],
    [0, 0, 0, 1]
])
  
ROT_TOTAL = T @ ROT_Z @ T_inv

3D matrixes to at3d scatterer:

In [9]:
DATA_DICT = OrderedDict({'density':lwc,'reff':reff,'veff':veff})

NEW_DATA_DICT = OrderedDict() # after crop and pad

# set grid using new pyshdom:
# make a grid for microphysics which is just the cloud grid.
cloud_scatterer = at3d.grid.make_grid(dx,nx,\
                          dy,ny,zgrid)

#------------------------------------------------------------------------
#------------------------------------------------------------------------
#------------------------------------------------------------------------

# original
non_zero_indexes = np.where(DATA_DICT['density']>0)
i, j, k = non_zero_indexes

for data_name in ('density' , 'reff', 'veff'):
    field = DATA_DICT[data_name]
    
    NEW_DATA_DICT[data_name] = field
    #initialize with np.nans so that empty data is np.nan    
    this_data = np.zeros((cloud_scatterer.sizes['x'], \
                cloud_scatterer.sizes['y'], cloud_scatterer.sizes['z']))*np.nan
    this_data[i, j, k] = field[i, j, k]
    cloud_scatterer[data_name] = (['x', 'y', 'z'], this_data)
    
#--------------------------------------
#--------------------------------------
#--------------------------------------




Visualization

In [11]:
mlab.figure(size=(600, 600))
figh = mlab.gcf()

# show_scatterer(cloud_scatterer)
mlab.orientation_axes()
mlab.quiver3d(sun_virtual_position[0],sun_virtual_position[1],sun_virtual_position[2],\
              sun_direction[0], sun_direction[1], sun_direction[2], line_width=8,color = (1.0, 1.0, 0), scale_factor=1)

#mlab.show()

ImportError: Could not import backend for traitsui.  Make sure you
        have a suitable UI toolkit like PyQt/PySide or wxPython
        installed.

# Prepare formations:

In [9]:
#--------------------------------------------
#--------------------------------------------
#--------------------------------------------
# ------------ Prepare sensors:--------------
#--------------------------------------------
sensor_dict = at3d.containers.SensorsDict()
transformed_sensor_dict = at3d.containers.SensorsDict()

# prepare views:
sat_positions, near_nadir_view_index, theta_max, theta_min = \
    StringOfPearls(SATS_NUMBER = SATS_NUMBER_SETUP,\
    orbit_altitude = Rsat,\
    move_nadir_x=CENTER_OF_MEDIUM_BOTTOM[0],\
    move_nadir_y=CENTER_OF_MEDIUM_BOTTOM[1])

names = ["sat"+str(i+1) for i in range(len(sat_positions))] 
# we intentialy, work with projections lists.
up_list = np.array(len(sat_positions)*[0,1,0]).reshape(-1,3) # default up vector per camera.

#------------------------------------------------------------------------------------
#------------------------------------------------------------------------------------
#------------------------------------------------------------------------------------
for position_vector,lookat_vector,up_vector,name in zip(sat_positions,\
                              SAT_LOOKATS,up_list,names):

    loop_sensor = at3d.sensor.perspective_projection(wavelength = wavelengths_micron, fov = fov,\
    x_resolution = cnx, y_resolution = cny,\
    position_vector = position_vector, lookat_vector = lookat_vector,\
    up_vector = up_vector, stokes=['I','Q','U'], sub_pixel_ray_args={'method':at3d.sensor.stochastic,'nrays':1})


    sensor_dict.add_sensor('CloudCT', loop_sensor)

    
sensor_list = sensor_dict['CloudCT']['sensor_list']
#------------------------------------------------------------------------------------
#------------------------------------------------------------------------------------
#------------------------------------------------------------------------------------
for sensor,lookat_vector,up_vector,name in zip(sensor_list,\
                              SAT_LOOKATS,up_list,names):

    
    new_position = np.dot( ROT_TOTAL, np.append(sensor.position,1) )
    new_rotation_matrix = np.dot( ROT_Z[0:3,0:3], sensor.rotation_matrix.reshape(3,-1))

    loop_sensor = at3d.sensor.perspective_projection(wavelength = wavelengths_micron, fov = fov,\
    x_resolution = cnx, y_resolution = cny,\
    position_vector = new_position[0:3], lookat_vector = lookat_vector,\
    up_vector = np.dot( ROT_Z[0:3,0:3],up_vector), stokes=['I','Q','U'], sub_pixel_ray_args={'method':at3d.sensor.stochastic,'nrays':1})

    transformed_sensor_dict.add_sensor('CloudCT', loop_sensor)
    
    condition = np.all(np.isclose(loop_sensor.rotation_matrix.reshape((3,-1)),\
               new_rotation_matrix, atol=1e-05))
    assert condition, "inconsistency in rotation."
    
transformed_sensor_list = transformed_sensor_dict['CloudCT']['sensor_list']


Satellites angles are:
[ 39.33903712  31.33231386  21.96378374  11.38075698   0.05729576
 -11.27042431 -21.86457631 -31.24752224 -39.26878875 -46.00712309]
max angle 0.058215689128220054
min angle -0.07276961141027506



Visualization

In [ ]:
mlab.figure(size=(600, 600))
figh = mlab.gcf()
at3d.sensor.show_sensors(sensor_list, scale = 50, axisWidth = 1.0, axisLenght=50, Show_Rays =  False, FullCone = True)

at3d.sensor.show_sensors(transformed_sensor_list, scale = 50, axisWidth = 2.0, axisLenght=60, Show_Rays =  False, FullCone = True)

mlab.orientation_axes()

sun_virtual_position = -sun_direction*500

mlab.quiver3d(sun_virtual_position[0],sun_virtual_position[1],sun_virtual_position[2],\
              sun_direction[0], sun_direction[1], sun_direction[2], line_width=8,color = (1.0, 1.0, 0), scale_factor=100)

# mlab.show() # commenc all mlab.show() when using "qt"

XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2


In [11]:
# mlab.show()

# Simulating Radiances

In [ ]:
# -----------------------------------------------------
# -----------------------------------------------------
# --------Simulating Radiances-------------------------
# -----------------------------------------------------
#load atmosphere
atmosphere = xr.open_dataset('../data/ancillary/AFGL_summer_mid_lat.nc')
#subset the atmosphere, choose only the bottom four km.
reduced_atmosphere = atmosphere.sel({'z': atmosphere.coords['z'].data[atmosphere.coords['z'].data <= 4.0]})
#merge the atmosphere and cloud z coordinates
merged_z_coordinate = at3d.grid.combine_z_coordinates([reduced_atmosphere,cloud_scatterer])    

# -----------------------------------------------------
# make a grid for microphysics which is just the cloud grid.
rte_grid = at3d.grid.make_grid(dx,cloud_scatterer.x.data.size,
                               dy,cloud_scatterer.y.data.size,
                          merged_z_coordinate)

cloud_scatterer_on_rte_grid = at3d.grid.resample_onto_grid(rte_grid, cloud_scatterer)

# -----------------------------------------------------
# -----------------------------------------------------

if (0):
    mlab.figure(size=(600, 600))
    figh = mlab.gcf()



    data_vis = cloud_scatterer_on_rte_grid.density.data[...,0:nz]
    src = mlab.pipeline.scalar_field(X, Y, Z, data_vis, figure=figh)
    src.spacing = [dx, dy, dz]
    src.update_image_data = True

    isosurface = mlab.pipeline.iso_surface(src, contours=[0.1*data_vis.max(),\
                                                          0.2*data_vis.max(),\
                                                          0.3*data_vis.max(),\
                                                          0.4*data_vis.max(),\
                                                          0.5*data_vis.max(),\
                                                          0.6*data_vis.max(),\
                                                          0.7*data_vis.max(),\
                                                          0.8*data_vis.max(),\
                                                          0.9*data_vis.max(),\
                                                          ],opacity=0.9,figure=figh)


    mlab.orientation_axes()
    mlab.outline(figure=figh,color = (1, 1, 1))


    mlab.figure(size=(600, 600))
    figh = mlab.gcf()


    data_vis = cloud_scatterer_on_rte_grid.density.data[...,0:nz]
    src = mlab.pipeline.scalar_field(X, Y, Z, data_vis, figure=figh)
    src.spacing = [dx, dy, dz]
    src.update_image_data = True

    isosurface = mlab.pipeline.iso_surface(src, contours=[0.1*data_vis.max(),\
                                                          0.2*data_vis.max(),\
                                                          0.3*data_vis.max(),\
                                                          0.4*data_vis.max(),\
                                                          0.5*data_vis.max(),\
                                                          0.6*data_vis.max(),\
                                                          0.7*data_vis.max(),\
                                                          0.8*data_vis.max(),\
                                                          0.9*data_vis.max(),\
                                                          ],opacity=0.9,figure=figh)

    mlab.orientation_axes()
    mlab.outline(figure=figh,color = (1, 1, 1))

    mlab.show()
# -----------------------------------------------------
# -----------------------------------------------------
# -----------------------------------------------------
# -----------------------------------------------------
# -----------------------------------------------------
# -----------------------------------------------------
# -----------------------------------------------------
# We choose a gamma size distribution and therefore need to define a 'veff' variable.
size_distribution_function = at3d.size_distribution.gamma

wavelengths = sensor_dict.get_unique_solvers()
wavelength_band = (wavelengths[0], wavelengths[0])

wavelen1, wavelen2 = wavelength_band
wavelength_averaging = False
formatstr = 'TEST_Water_{}nm.nc'.format(int(1e3*wavelength_band[0]))
safe_mkdirs('../mie_tables')
if not (wavelen1 == wavelen2):
    wavelength_averaging = True   
    formatstr = 'TEST_averaged_Water_{}-{}nm.nc'.format(int(1e3*wavelength_band[0]), int(1e3*wavelength_band[1]))
mono_path = os.path.join('../mie_tables', formatstr)


# Exact OpticalPropertyGenerator:
# get_mono_table will first search a directory to see if the requested table exists otherwise it will calculate it. 
# You can save it to see if it works.
mie_mono_table = at3d.mie.get_mono_table(
    'Water',wavelength_band,
    max_integration_radius=65.0,
    minimum_effective_radius=0.1,
    relative_dir='../mie_tables',
    verbose=False
)
    
mie_mono_table.to_netcdf(mono_path)
mie_mono_tables = OrderedDict()
mie_mono_tables[wavelength_band[0]] = mie_mono_table

optical_prop_gen = at3d.medium.OpticalPropertyGenerator(
    'cloud',
    mie_mono_tables, 
    size_distribution_function,
    particle_density=1.0, 
    maxnphase=None,
    interpolation_mode='exact',
    density_normalization='density',#The density_normalization argument is a convenient
    reff=np.linspace(1,30.0,50),
    veff=np.linspace(0.01,0.15,15)
)

optical_properties = optical_prop_gen(cloud_scatterer_on_rte_grid)

# If you generate your own optical properties they must pass this check to be used in the solver.
at3d.checks.check_optical_properties(optical_properties[wavelength_band[0]])

# one function to generate rayleigh scattering.
rayleigh_scattering = at3d.rayleigh.to_grid(wavelengths,atmosphere,rte_grid)    


# -----------------------------------------------------
# -----------------------------------------------------
# -----------------------------------------------------
# -----------------------------------------------------
# -----------------------------------------------------
# -----------------------------------------------------
# -----------------------------------------------------
# Define Solvers:

solvers_dict = at3d.containers.SolversDict()
transformed_solvers_dict = at3d.containers.SolversDict()
# note we could set solver dependent surfaces / sources / numerical_config here
# just as we have got solver dependent optical properties.
# original:
for wavelength in wavelengths:
    medium = {
        'cloud': optical_properties[wavelength],
        'rayleigh':rayleigh_scattering[wavelength]
     }
    config = at3d.configuration.get_config()
    solvers_dict.add_solver(
        wavelength,
        at3d.solver.RTE(
            numerical_params=config,
            surface=at3d.surface.lambertian(0.05),
            source=source,
            medium=medium,
            num_stokes=3#sensor_dict.get_minimum_stokes()[wavelength],
        )                   
 )
# transformed: use same cloud and optical properties (only satellites are rotated)
for wavelength in wavelengths:
    medium = {
        'cloud': optical_properties[wavelength],
        'rayleigh':rayleigh_scattering[wavelength]
     }
    config = at3d.configuration.get_config()
    transformed_solvers_dict.add_solver(
        wavelength,
        at3d.solver.RTE(
            numerical_params=config,
            surface=at3d.surface.lambertian(0.05),
            source=source,
            medium=medium,
            num_stokes=3#sensor_dict.get_minimum_stokes()[wavelength],
        )                   
 )
        
# solve the 4 RTEs in parallel AND get the measurements.
sensor_dict.get_measurements(solvers_dict, n_jobs=n_jobs, verbose=True)
transformed_sensor_dict.get_measurements(transformed_solvers_dict, n_jobs=n_jobs, verbose=True)



adapt_grid_factor reduced to  3.5604165301985304
adapt_grid_factor reduced to  3.5604165301985304
  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Polarization 0.672 micron]
     1  -0.762  0.225E+00    90000    15.86  0.062   [Polarization 0.672 micron]
     2  -1.754  0.229E+00    90000    15.86  0.062   [Polarization 0.672 micron]
     3  -1.885  0.208E+00    90146    16.24  0.063   [Polarization 0.672 micron]
     4  -1.935  0.121E+00    90824    17.90  0.070   [Polarization 0.672 micron]
     5  -1.725  0.891E-01    91759    19.77  0.077   [Polarization 0.672 micron]
     6  -2.029  0.880E-01    91772    19.80  0.077   [Polarization 0.672 micron]
     7  -2.162  0.713E-01    92424    21.23  0.083   [Polarization 0.672 micron]
     8  -2.193  0.589E-01    93471    22.85  0.089   [Polarization 0.672 micron]
     9  -2.125  0.523E-01    94069    23.72  0.093   [Polarization 0.672 micron]
    10  -2.362  0.516E-01    94140    23.86  0.093   [Polarization 0.672 micron]
    11  -2.343

AttributeError: 'collections.OrderedDict' object has no attribute 'extinction'

In [ ]:
optical_properties = optical_prop_gen(cloud_scatterer_on_rte_grid)[wavelength_band[0]]
extinction = optical_properties.extinction

tau = np.sum(extinction[...,0:nz]*dz, axis=2)


# Show images:

In [13]:
sensor_images_original = sensor_dict.get_images('CloudCT')
sensor_images_trans = transformed_sensor_dict.get_images('CloudCT')

In [14]:
PNCHANNELS = 1 # polarized channels
pol_channels = ['I']
if 'Q' in list(sensor_images_original[0].keys()) and 'U' in list(sensor_images_original[0].keys()):
    print(" The images are polarized")
    PNCHANNELS = 3
    pol_channels = ['I','Q','U']
    
nrows = 3
LN = len(sensor_images_original)
ncols = int(LN) 

 The images are polarized


In [15]:
# Create the kernel
kernel_size=3
kernel = np.ones((kernel_size, kernel_size), dtype=np.uint8)
# ------------------------------
fontsize = 16
for pol_channel in pol_channels:
    
#     if not pol_channel == 'I':
#         continue
        
    fig = plt.figure(figsize=(20, 10))
    fontsize = 16
    fig.subplots_adjust(hspace=0.4, wspace=0.4) 
    max_ = 0
    min_ = 1e7
    
    for out_index,sensor_images in enumerate([sensor_images_original,sensor_images_trans]):
        for index,sensor in enumerate(sensor_images):
            img = sensor[pol_channel].T.data
            max_ = max(max_,img.max())
            min_ = min(min_,img.min())
        max_ = max(max_,img.max())
        min_ = min(min_,img.min())                
    cmap = 'gray'
    
    for index in range(LN):
        img1 = sensor_images_original[index][pol_channel].T.data
        img2 = sensor_images_trans[index][pol_channel].T.data
        
        if pol_channel == 'I':
            min_ = 0
            cmap = 'gray'
        #1:
        ii = index+1 + (ncols*0)
        ax = fig.add_subplot(nrows, ncols, ii)
        im = ax.imshow( img1 ,cmap=cmap,vmin = min_, vmax = max_)
        title = "{}".format(index)
        ax.set_title(title, fontsize=fontsize)
        
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.01)
        plt.colorbar(im, cax=cax)                 
#         ax.set_axis_off()
        
        #2:
        ii = index+1 + (ncols*1)
        ax = fig.add_subplot(nrows, ncols, ii)
        im = ax.imshow( img2 ,cmap=cmap,vmin = min_, vmax = max_)
        title = "{}".format(index)
        ax.set_title(title, fontsize=fontsize)    
        
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.01)
        plt.colorbar(im, cax=cax)                 
        ax.set_axis_off()
        
        # diff:
        err = 100*np.abs(img1-img2)/np.abs(img1)
        if pol_channel == 'I':
            mask = np.zeros_like(img1)
            mask[img1 > 0.02] = 1
            # Perform erosion
            eroded_mask = binary_erosion(mask, structure=kernel, iterations=2)

        else:
            mask = np.zeros_like(img1)
            mask[sensor_images_original[index]['I'].T.data > 0.02] = 1
            # Perform erosion
            eroded_mask = binary_erosion(mask, structure=kernel, iterations=2)

        ii = index+1 + (ncols*2)
        ax = fig.add_subplot(nrows, ncols, ii)
        im = ax.imshow( err*eroded_mask ,cmap=cmap)
        title = "diff"
        ax.set_title(title, fontsize=fontsize)
        
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.01)
        plt.colorbar(im, cax=cax)                 
        ax.set_axis_off()

    fig.suptitle("channel {}".format(pol_channel), size=16,y=0.95)

     

XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2
XmbTextListToTextProperty result code -2


In [31]:
from scipy.ndimage import binary_erosion

